In [15]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
from collections import Counter
import os
from pathlib import Path

In [16]:
# -----------------------------
# CONFIG
# -----------------------------
REPO_ROOT = Path(os.getcwd()).parent 

VIDEO_PATH = "/work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4"
OUTPUT_VIDEO = "labeled_output.mp4"

ROBOT_MODEL_PATH = REPO_ROOT / "yolov8_model" / "best_tuned_yolov8.pt"
NUMBER_MODEL_PATH = REPO_ROOT / "number_reading" / "best_number.pt"

ROBOT_CLASS_ID = 1
RED_NUMBER_CLASS_ID = 1
BLUE_NUMBER_CLASS_ID = 0

FRAME_SKIP = 15

In [17]:
# -----------------------------
# LOAD MODELS
# -----------------------------
robot_model = YOLO(str(ROBOT_MODEL_PATH))
number_model = YOLO(str(NUMBER_MODEL_PATH))

reader = easyocr.Reader(['en'], gpu=True)

In [18]:
# -----------------------------
# IMAGE PROCESSING
# -----------------------------
def estimate_angle_from_crop(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return 0.0

    cnt = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(cnt)
    angle = rect[-1]

    if angle < -45:
        angle += 90

    return angle


def rotate_image(img, angle):
    h, w = img.shape[:2]
    center = (w // 2, h // 2)

    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )

def preprocess_variants(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    variants = []

    # Threshold sweep
    for t in [150, 165, 180, 195, 210]:
        _, th = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
        variants.append(th)

    # Adaptive
    adaptive = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )
    variants.append(adaptive)

    # Otsu
    _, otsu = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    variants.append(otsu)

    return variants


def read_number_from_image(img):
    variants = preprocess_variants(img)
    guesses = []

    for processed in variants:
        angle = estimate_angle_from_crop(img)
        rotated = rotate_image(processed, angle)

        results = reader.readtext(
            rotated,
            allowlist="0123456789",
            detail=1,
            paragraph=False
        )

        results = [r for r in results if r[2] > 0.35]

        if results:
            guesses.append(results[0][1])

    if not guesses:
        return None

    c = Counter(guesses)
    max_count = max(c.values())
    tied = [num for num, count in c.items() if count == max_count]

    return max(tied, key=lambda x: len(str(x)))

In [19]:
# -----------------------------
# MATCHING SYSTEM
# -----------------------------
def levenshtein(a, b):
    a, b = str(a), str(b)
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]

    for i in range(len(a)+1):
        dp[i][0] = i
    for j in range(len(b)+1):
        dp[0][j] = j

    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[-1][-1]


def similarity(a, b):
    if not a or not b:
        return 0
    dist = levenshtein(a, b)
    return 1 - dist / max(len(str(a)), len(str(b)))


def alignment_score(a, b):
    a, b = str(a), str(b)
    best = 0

    for shift in range(-len(b), len(a)+1):
        matches = 0
        for i in range(len(a)):
            j = i - shift
            if 0 <= j < len(b) and a[i] == b[j]:
                matches += 1
        best = max(best, matches)

    return best / max(len(a), len(b))


def combined_score(a, b, w1=0.7, w2=0.3):
    return w1 * similarity(a, b) + w2 * alignment_score(a, b)


def match_number_single(detected, nums):
    if detected is None:
        return None

    best_score = 0.5
    match = None

    for n in nums:
        current = combined_score(detected, n)
        if current > best_score:
            best_score = current
            match = n

    return match

In [20]:
# -----------------------------
# MAIN
# -----------------------------
def process_video(video_path, red_team_numbers=[], blue_team_numbers=[], crop=True):
    cap = cv2.VideoCapture(video_path)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = None

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        if frame_idx % FRAME_SKIP != 0:
            continue

        h, w = frame.shape[:2]

        if crop:
            frame = frame[int(h * 3 / 5):h, 0:w]
        
        if out is None:
            h2, w2 = frame.shape[:2]

            fourcc = cv2.VideoWriter_fourcc(*"XVID")
            fps = cap.get(cv2.CAP_PROP_FPS) / FRAME_SKIP

            out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w2, h2))

        robot_results = robot_model(frame, verbose=False)[0]

        for box in robot_results.boxes:
            if int(box.cls[0]) != ROBOT_CLASS_ID:
                continue

            if box.conf[0] < 0.3:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            robot_crop = frame[y1:y2, x1:x2]

            if robot_crop.size == 0:
                continue

            label_text = "Unknown"
            color = (0, 0, 255)

            best_match = None
            best_score = 0

            number_results = number_model(robot_crop, verbose=False)[0]

            for nbox in number_results.boxes:

                cls_id = int(nbox.cls[0])
                if cls_id == RED_NUMBER_CLASS_ID:
                    team_list = red_team_numbers
                elif cls_id == BLUE_NUMBER_CLASS_ID:
                    team_list = blue_team_numbers
                else:
                    continue


                if nbox.conf[0] < 0.3:
                    continue

                nx1, ny1, nx2, ny2 = map(int, nbox.xyxy[0])

                pad = 10
                h2, w2 = robot_crop.shape[:2]

                nx1 = max(0, nx1 - pad)
                ny1 = max(0, ny1 - pad)
                nx2 = min(w2, nx2 + pad)
                ny2 = min(h2, ny2 + pad)

                if (nx2 - nx1) < 30 or (ny2 - ny1) < 15:
                    continue

                number_crop = robot_crop[ny1:ny2, nx1:nx2]

                detected = read_number_from_image(number_crop)
                matched = match_number_single(detected, team_list)

                if matched is not None:
                    score = combined_score(detected, matched)

                    # KEEP BEST MATCH ONLY
                    if score > best_score:
                        best_score = score
                        best_match = matched
                        label_text = str(matched)
                        color = (0, 255, 0)

                elif detected is not None and best_match is None:
                    # Only use fallback if no good match yet
                    label_text = f"?{detected}"
                    color = (0, 255, 255)

            # Draw box + label
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            cv2.putText(
                frame,
                label_text,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                color,
                2,
                cv2.LINE_AA
            )

        out.write(frame)

        print(f"Frame {frame_idx} processed")

    cap.release()
    if out:
        out.release()

In [21]:
# -----------------------------
# RUN
# -----------------------------

RED_TEAM_NUMBERS = [8825, 1736, 2357]
BLUE_TEAM_NUMBERS = [5809, 9570, 3928]

if __name__ == "__main__":
    process_video(VIDEO_PATH, RED_TEAM_NUMBERS, BLUE_TEAM_NUMBERS, False)

OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'


Frame 15 processed
Frame 30 processed
Frame 45 processed
Frame 60 processed
Frame 75 processed
Frame 90 processed
Frame 105 processed
Frame 120 processed
Frame 135 processed
Frame 150 processed
Frame 165 processed
Frame 180 processed
Frame 195 processed
Frame 210 processed
Frame 225 processed
Frame 240 processed
Frame 255 processed
Frame 270 processed
Frame 285 processed
Frame 300 processed
Frame 315 processed
Frame 330 processed
Frame 345 processed
Frame 360 processed
Frame 375 processed
Frame 390 processed
Frame 405 processed
Frame 420 processed
Frame 435 processed
Frame 450 processed
Frame 465 processed
Frame 480 processed
Frame 495 processed
Frame 510 processed
Frame 525 processed
Frame 540 processed
Frame 555 processed
Frame 570 processed
Frame 585 processed
Frame 600 processed
Frame 615 processed
Frame 630 processed
Frame 645 processed
Frame 660 processed
Frame 675 processed
Frame 690 processed
Frame 705 processed
Frame 720 processed
Frame 735 processed
Frame 750 processed
Frame 